# Eksplorasi Data API & Pembersihan Menggunakan Pandas

Notebook ini dirancang sebagai **pembanding dari versi Polars (Pertemuan 2)**. Di sini kita akan melihat bagaimana alur kerja yang sama diselesaikan dengan **Pandas** (library DataFrame klasik):
1. Mengambil data produk dari **DummyJSON API**.
2. Memuat data JSON tersebut ke **Pandas DataFrame**.
3. Memahami karakteristik **Eager Evaluation** pada Pandas (tidak ada mode Lazy).
4. Menggunakan operasi Pandas untuk pembersihan data, pengisian null, dan filter.
5. Menyimpan data akhir ke format **Parquet**.

## 🛠️ 1. Import Library & Setup

In [1]:
import pandas as pd
import requests
import json

print(f"Pandas version: {pd.__version__}")

Pandas version: 2.0.3


## 📥 2. Extract: Ambil Data dari DummyJSON API

In [2]:
api_url = "https://dummyjson.com/products"

try:
    print(f"Mencoba mengambil data dari: {api_url}...")
    response = requests.get(api_url, timeout=5)
    response.raise_for_status()
    raw_data = response.json()
    print(f"✅ Sukses! Ditemukan {len(raw_data['products'])} data dari API.")
except Exception as e:
    print(f"⚠️ Gagal mengakses API: {e}")
    print("🔄 Menggunakan file fallback lokal 'data_dummy.json'...")
    with open("../data_dummy.json", "r", encoding="utf-8") as f:
        raw_data = json.load(f)
    print(f"✅ Sukses memuat {len(raw_data['products'])} data dari file fallback.")

products_list = raw_data["products"]

Mencoba mengambil data dari: https://dummyjson.com/products...
✅ Sukses! Ditemukan 30 data dari API.


## 📊 3. Load ke Pandas DataFrame (Eager Mode)

Pandas langsung membuat DataFrame di memori secara Eager.

In [3]:
df = pd.DataFrame(products_list)
print(df[['id', 'title', 'price', 'stock', 'category', 'rating']].head(3))

   id                          title  price  stock category  rating
0   1  Essence Mascara Lash Princess   9.99    5.0   beauty    4.94
1   2  Eyeshadow Palette with Mirror  19.99    NaN   beauty    3.50
2   3                Powder Canister    NaN   10.0   beauty    4.00


## ⚡ 4. Eager Evaluation di Pandas

Berbeda dengan Polars, Pandas **tidak memiliki mode Lazy**. Setiap fungsi yang kita panggil akan langsung dihitung di memori dan memakan daya komputasi di saat itu juga. 
Tidak ada pengoptimalan rencana query (*no query optimization*) sebelum eksekusi.

## 🧹 5. Transform: Pembersihan Data Menggunakan Pandas

Mari kita lakukan transformasi yang sama:
1.  **Handling Nulls**: Mengisi `price` null dengan median, `stock` null dengan `0`, `category` null dengan `'other'`.
2.  **Filtering**: Mengambil rating >= `4.0`.
3.  **Renaming & Formatting**: Kolom `title` di-upper case dan di-alias menjadi `nama_produk`.

In [4]:
# 1. Mengisi null & casting
df['price'] = df['price'].fillna(df['price'].median())
df['stock'] = df['stock'].fillna(0).astype('int64')
df['category'] = df['category'].fillna('other')
df['rating'] = df['rating'].fillna(0.0)

# 2. Filter rating >= 4.0
df_filtered = df[df['rating'] >= 4.0].copy()

# 3. Rename & Format Text
df_filtered = df_filtered.rename(columns={
    'id': 'id_produk',
    'title': 'nama_produk',
    'price': 'harga',
    'stock': 'stok',
    'category': 'kategori',
    'rating': 'skor_rating'
})

df_filtered['nama_produk'] = df_filtered['nama_produk'].str.upper()

# 4. Memilih kolom akhir
final_df = df_filtered[['id_produk', 'nama_produk', 'harga', 'stok', 'kategori', 'skor_rating']]

print("Pembersihan data versi Pandas selesai.")

Pembersihan data versi Pandas selesai.


## 🚀 6. Menampilkan Hasil DataFrame Pandas

In [5]:
print(final_df.head(3))

   id_produk                    nama_produk  harga  stok kategori  skor_rating
0          1  ESSENCE MASCARA LASH PRINCESS   9.99     5   beauty         4.94
2          3                POWDER CANISTER  11.49    10   beauty         4.00
3          4                   RED LIPSTICK  12.99     0    other         4.80


## 💾 7. Load: Menyimpan ke Format Parquet

In [6]:
output_path = "products_cleaned_pandas.parquet"
final_df.to_parquet(output_path, engine='pyarrow')
print(f"✅ Data Pandas berhasil disimpan ke: {output_path}")

✅ Data Pandas berhasil disimpan ke: products_cleaned_pandas.parquet
